<a href="https://colab.research.google.com/github/niniqoiii/Springfield/blob/main/inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install -q gdown

import os
import json
import zipfile
import shutil
import random
import gdown
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms


In [9]:
file_id = "12FqLnmk6Cu0NR8UehRQiD7az2iDYufCX"

zip_file = "/content/Simpsons.zip"
extract_dir = "/content/simpsons_extracted"

if not os.path.exists(zip_file):
    gdown.download(f"https://drive.google.com/uc?id={file_id}", zip_file, quiet=False)

if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print("Dataset extracted")


Dataset extracted


In [10]:
train_dir = None

for root, dirs, files in os.walk(extract_dir):
    if "characters_train" in dirs and "__MACOSX" not in root:
        train_dir = os.path.join(root, "characters_train")
        break

print("Train dir:", train_dir)


Train dir: /content/simpsons_extracted/archive/characters_train


In [11]:
random.seed(42)

test_dir = "/content/test_images"

if os.path.exists(test_dir):
    shutil.rmtree(test_dir)

os.makedirs(test_dir, exist_ok=True)

classes_all = sorted([
    d for d in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, d))
])

selected_classes = random.sample(classes_all, min(15, len(classes_all)))

copied = 0

for cls in selected_classes:

    cls_path = os.path.join(train_dir, cls)

    images = [
        f for f in os.listdir(cls_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    random.shuffle(images)

    for img in images[:5]:

        src = os.path.join(cls_path, img)

        dst = os.path.join(test_dir, f"{cls}_{img}")

        shutil.copy(src, dst)

        copied += 1

print(f"Created random test_images folder with {copied} images")
print("Classes used:", selected_classes)


Created random test_images folder with 75 images
Classes used: ['troy_mcclure', 'chief_wiggum', 'agnes_skinner', 'krusty_the_clown', 'homer_simpson', 'groundskeeper_willie', 'cletus_spuckler', 'charles_montgomery_burns', 'carl_carlson', 'moe_szyslak', 'apu_nahasapeemapetilon', 'abraham_grampa_simpson', 'patty_bouvier', 'rainier_wolfcastle', 'waylon_smithers']


In [12]:
# ============================================================================
# Model Definition — must match train.ipynb exactly
# ============================================================================
# FIX: The original inference notebook used a different class called SimpleCNN
# which had a different architecture (Dropout 0.5 vs 0.4) from what was
# actually trained (SimpsonsCNN). Using the wrong architecture causes a
# state_dict key mismatch or silent weight corruption.
# This is now corrected to use SimpsonsCNN, matching train.ipynb exactly.

class SimpsonsCNN(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),       # 64 -> 32

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),       # 32 -> 16

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),       # 16 -> 8

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),       # 8 -> 4
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x


In [13]:
def infer(data_dir, model_path):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    with open("classes.json", "r") as f:
        classes = json.load(f)

    # FIX: Use SimpsonsCNN (the architecture actually trained), not SimpleCNN.
    model = SimpsonsCNN(len(classes))

    model.load_state_dict(torch.load(model_path, map_location=device))

    model.to(device)

    model.eval()

    # =========================================================================
    # FIX (critical): The original inference notebook was MISSING normalization.
    # Training used Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]).
    # Without the same normalization at inference, raw pixel values fall far
    # outside the distribution the model learned, causing it to collapse to
    # always predicting the most common class (homer_simpson).
    # The transform here now exactly mirrors the val_transform from train.ipynb.
    # =========================================================================
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])   # <-- was missing!
    ])

    results = {}

    for file in sorted(os.listdir(data_dir)):

        if file.lower().endswith((".jpg", ".jpeg", ".png")):

            path = os.path.join(data_dir, file)

            image = Image.open(path).convert("RGB")

            image = transform(image).unsqueeze(0).to(device)

            with torch.no_grad():

                outputs = model(image)
                probs = torch.softmax(outputs, dim=1)[0]

                predicted = torch.argmax(probs).item()

                # Top-3 predictions with confidence scores
                top3_probs, top3_indices = torch.topk(probs, k=3)
                top3 = [
                    {"class": classes[idx.item()], "confidence": round(prob.item(), 4)}
                    for prob, idx in zip(top3_probs, top3_indices)
                ]

            results[file] = {
                "predicted": classes[predicted],
                "top3": top3
            }

    with open("results.json", "w") as f:
        json.dump(results, f, indent=4)

    print(f"results.json created with {len(results)} predictions")

    return results


In [14]:
model_path = "model.pth"

results = infer(test_dir, model_path)

# Show predictions: filename -> predicted class
list((fname, v["predicted"]) for fname, v in list(results.items())[:100])


results.json created with 75 predictions


[('abraham_grampa_simpson_pic_0223.jpg', 'abraham_grampa_simpson'),
 ('abraham_grampa_simpson_pic_0364.jpg', 'apu_nahasapeemapetilon'),
 ('abraham_grampa_simpson_pic_0446.jpg', 'abraham_grampa_simpson'),
 ('abraham_grampa_simpson_pic_0505.jpg', 'abraham_grampa_simpson'),
 ('abraham_grampa_simpson_pic_0661.jpg', 'abraham_grampa_simpson'),
 ('agnes_skinner_pic_0001.jpg', 'kent_brockman'),
 ('agnes_skinner_pic_0002.jpg', 'agnes_skinner'),
 ('agnes_skinner_pic_0018.jpg', 'agnes_skinner'),
 ('agnes_skinner_pic_0026.jpg', 'agnes_skinner'),
 ('agnes_skinner_pic_0031.jpg', 'agnes_skinner'),
 ('apu_nahasapeemapetilon_pic_0001.jpg', 'apu_nahasapeemapetilon'),
 ('apu_nahasapeemapetilon_pic_0090.jpg', 'apu_nahasapeemapetilon'),
 ('apu_nahasapeemapetilon_pic_0398.jpg', 'apu_nahasapeemapetilon'),
 ('apu_nahasapeemapetilon_pic_0435.jpg', 'apu_nahasapeemapetilon'),
 ('apu_nahasapeemapetilon_pic_0484.jpg', 'apu_nahasapeemapetilon'),
 ('carl_carlson_pic_0008.jpg', 'carl_carlson'),
 ('carl_carlson_pic_00